In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

**⚠️ Data Requirement — BioImageArchive:** This notebook requires raw microscopy data from BioImageArchive. Download the dataset and set `data_archive_path` to your local BioImageArchive directory (see README).

**Pipeline step 1/2** — run before `1_fluo_binary_metabolic_gradient.ipynb`.

**Run analysis scripts in this order:**
1. `0_combined_df_metabolic_gradient.ipynb` — load and combine raw image data
2. `1_fluo_binary_metabolic_gradient.ipynb` — extract fluorescence binary data

# Combined DataFrame - Metabolic Gradient

This notebook aggregates fluorescence intensity data from multiple replicates and positions for the metabolic gradient experiment.

**Input:** Raw imaging data from BioImage Archive (`SAGrowth/S7_wt_SA-growth-estimation/`)

**Output:** `analysis_code/0_combined_df_metabolic_gradient.csv`

**Steps:**
1. Iterates through all replicate folders in the Image_Data directory (data available on BioImageArchive)
2. Within each replicate, searches for position folders and reads `single_cell_props.csv`
3. Adds metadata columns: `strain` (set to `'wt'`), `replicate`, and `pos`
4. Concatenates all dataframes and saves as `0_combined_df_metabolic_gradient.csv`

In [3]:

# set path to BioImageArchive data directory:
data_archive_path = '/Volumes/ScientificData/Users/Giulia(botgiu00)/Papers/bottacin2026/BioImageArchive/'

# input and output paths relative to BioImageArchive location
input_dir = os.path.join(data_archive_path, 'SAGrowth/S7_wt_SA-growth-estimation')
output_file = '0_combined_df_metabolic_gradient.csv'

# List to store all dataframes
all_data = []

# Get sorted replicate folders
replicate_folders = sorted([f for f in os.listdir(input_dir) 
                           if os.path.isdir(os.path.join(input_dir, f)) 
                           and f.startswith('replicate')])

print(f"Found {len(replicate_folders)} replicates")

# Iterate through replicate folders
for replicate_folder in replicate_folders:
    replicate_path = os.path.join(input_dir, replicate_folder)
    
    # Get sorted position folders
    pos_folders = sorted([f for f in os.listdir(replicate_path) 
                         if f.startswith('pos')])
    
    # Iterate through position folders
    for pos_folder in pos_folders:
        csv_path = os.path.join(replicate_path, pos_folder, 'single_cell_props.csv')
        
        # Load data if file exists
        if os.path.isfile(csv_path):
            df = pd.read_csv(csv_path)
            
            # Add metadata columns
            df.insert(1, 'strain', 'wt')
            df.insert(2, 'replicate', replicate_folder)
            df.insert(3, 'pos', pos_folder)
            
            all_data.append(df)
            print(f"  Loaded {replicate_folder}/{pos_folder}: {len(df)} rows")

# Concatenate all dataframes and save
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    combined_df.to_csv(output_file, index=False)
    print(f"\n✅ Combined dataframe saved: {len(combined_df)} total rows, {len(all_data)} positions")
else:
    print("⚠️ No data found")
    combined_df = pd.DataFrame()

combined_df.head()

Found 1 replicates
  Loaded replicate_1/pos18: 776375 rows
  Loaded replicate_1/pos19: 775504 rows
  Loaded replicate_1/pos20: 771941 rows
  Loaded replicate_1/pos21: 784078 rows
  Loaded replicate_1/pos23: 799210 rows
  Loaded replicate_1/pos24: 802636 rows
  Loaded replicate_1/pos25: 795406 rows

✅ Combined dataframe saved: 5505150 total rows, 7 positions


,label,strain,replicate,pos,area,min_row,min_col,max_row,max_col,intensity_max,intensity_mean,intensity_min,minor_axis_length,major_axis_length,y,x,intensity_mcherry,intensity_raw_mcherry,frame_number
0,1,wt,replicate_1,pos18,67.0,1,33,10,42,626.0,518.791045,355.0,9.046765,9.568206,4.746269,37.358209,59.805970,518.791045,0
1,2,wt,replicate_1,pos18,51.0,1,46,9,54,734.0,569.176471,458.0,7.871687,8.262165,4.568627,49.294118,71.274510,569.176471,0
2,3,wt,replicate_1,pos18,48.0,1,70,9,78,731.0,619.291667,528.0,7.412835,8.332576,4.333333,73.104167,82.750000,619.291667,0
3,4,wt,replicate_1,pos18,81.0,1,127,10,137,731.0,582.222222,466.0,9.752324,10.614854,5.049383,131.456790,74.259259,582.222222,0
4,5,wt,replicate_1,pos18,60.0,1,387,9,395,657.0,512.433333,427.0,8.748333,8.748333,4.500000,390.500000,58.350000,512.433333,0
